In [2]:
import pandas as pd 
import numpy as np 
import nltk
import gensim
import re 

In [3]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
data=pd.read_csv('all_kindle_review.csv')

In [5]:
data

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000
...,...,...,...,...,...,...,...,...,...,...,...
11995,11995,2183,B001DUGORO,"[0, 0]",4,Valentine cupid is a vampire- Jena and Ian ano...,"02 28, 2014",A1OKS5Q1HD8WQC,lisa jon jung,jena,1393545600
11996,11996,6272,B002JCSFSQ,"[2, 2]",5,I have read all seven books in this series. Ap...,"05 16, 2011",AQRSPXLNEQAMA,TerryLP,Peacekeepers Series,1305504000
11997,11997,12483,B0035N1V7K,"[0, 1]",3,This book really just wasn't my cuppa. The si...,"07 26, 2013",A2T5QLT5VXOJAK,hwilson,a little creepy,1374796800
11998,11998,3640,B001W1XT40,"[1, 2]",1,"tried to use it to charge my kindle, it didn't...","09 17, 2013",A28MHD2DDY6DXB,"Allison A. Slater ""Gryphon50""",didn't work,1379376000


In [6]:
df=data[['reviewText','rating']]

In [7]:
df

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4
...,...,...
11995,Valentine cupid is a vampire- Jena and Ian ano...,4
11996,I have read all seven books in this series. Ap...,5
11997,This book really just wasn't my cuppa. The si...,3
11998,"tried to use it to charge my kindle, it didn't...",1


In [8]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [9]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [10]:
df['rating']=df['rating'].apply(lambda x :0 if x<3 else 1)

C:\Users\HP\AppData\Local\Temp\ipykernel_22380\2668202038.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating']=df['rating'].apply(lambda x :0 if x<3 else 1)


In [11]:
df['rating']

0        1
1        1
2        1
3        1
4        1
        ..
11995    1
11996    1
11997    1
11998    0
11999    1
Name: rating, Length: 12000, dtype: int64

In [12]:
df['rating'].shape

(12000,)

In [13]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

## Preprocessing

In [14]:
df['reviewText']=df['reviewText'].str.lower()

C:\Users\HP\AppData\Local\Temp\ipykernel_22380\2784960931.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].str.lower()


In [15]:
df['reviewText']

0        jace rankin may be short, but he's nothing to ...
1        great short read.  i didn't want to put it dow...
2        i'll start by saying this is the first of four...
3        aggie is angela lansbury who carries pocketboo...
4        i did not expect this type of book to be in li...
                               ...                        
11995    valentine cupid is a vampire- jena and ian ano...
11996    i have read all seven books in this series. ap...
11997    this book really just wasn't my cuppa.  the si...
11998    tried to use it to charge my kindle, it didn't...
11999    taking instruction is a look into the often hi...
Name: reviewText, Length: 12000, dtype: object

In [16]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

C:\Users\HP\AppData\Local\Temp\ipykernel_22380\3736800740.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
C:\Users\HP\AppData\Local\Temp\ipykernel_22380\3736800740.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
C:\Users\HP\AppData\Local\Temp\ipykernel_22380\3736800740.py:6: SettingWithCopyWarning:

In [17]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [18]:
from nltk.stem import WordNetLemmatizer
Lemmatizer=WordNetLemmatizer()

In [19]:
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([Lemmatizer.lemmatize(word) for word in x.split()]))

C:\Users\HP\AppData\Local\Temp\ipykernel_22380\2097318463.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:" ".join([Lemmatizer.lemmatize(word) for word in x.split()]))


In [20]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [21]:
df['reviewText'].shape

(12000,)

In [22]:
df.shape

(12000, 2)

In [23]:
from sklearn.model_selection import train_test_split

In [24]:
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20,
                                               random_state=42)

In [25]:
X_train

9182     looking forward book came double space every p...
11091    already owned book spouse forgot already part ...
6428     cool forgot request rate came make mine unreli...
288      short short story basically scene party one ni...
2626     secret service agent secrests even longer serv...
                               ...                        
11964    downloaded book reading review usually reading...
5191     far one hottest book ive ever gotten hand ondo...
5390     even though book free reservation based majori...
860      little mushy 34must take care woman folk34 cha...
7270     book good good set charaterswith background le...
Name: reviewText, Length: 9600, dtype: object

In [26]:
X_train.shape

(9600,)

In [27]:
X_test

1935         really great read wish would hope find author
6494     nope tried cant read take greatest delight del...
1720     story line drug like book much mystery fan wou...
9120     read several angel book one work didnt really ...
360      possibly worst book ever read beginning positi...
                               ...                        
1195     enjoyed read think fan humorous must err use l...
11877    pleasantly surprised book enjoyed m dubois tol...
5421     love best friend since 15 year old 30 he servi...
3855     fascinating book enough twist turn keep readin...
4414     plot noted publisher blurb publisher make fun ...
Name: reviewText, Length: 2400, dtype: object

In [28]:
X_test.shape

(2400,)

In [29]:
y_train

9182     1
11091    0
6428     1
288      0
2626     1
        ..
11964    0
5191     1
5390     0
860      1
7270     1
Name: rating, Length: 9600, dtype: int64

In [30]:
y_train.shape

(9600,)

In [31]:
y_test.shape

(2400,)

In [32]:
y_test

1935     1
6494     0
1720     0
9120     0
360      0
        ..
1195     1
11877    1
5421     1
3855     1
4414     1
Name: rating, Length: 2400, dtype: int64

In [33]:
X_train = X_train.apply(lambda x: x.split())
X_test = X_test.apply(lambda x: x.split())

In [34]:
X_train

9182     [looking, forward, book, came, double, space, ...
11091    [already, owned, book, spouse, forgot, already...
6428     [cool, forgot, request, rate, came, make, mine...
288      [short, short, story, basically, scene, party,...
2626     [secret, service, agent, secrests, even, longe...
                               ...                        
11964    [downloaded, book, reading, review, usually, r...
5191     [far, one, hottest, book, ive, ever, gotten, h...
5390     [even, though, book, free, reservation, based,...
860      [little, mushy, 34must, take, care, woman, fol...
7270     [book, good, good, set, charaterswith, backgro...
Name: reviewText, Length: 9600, dtype: object

In [35]:
X_test

1935     [really, great, read, wish, would, hope, find,...
6494     [nope, tried, cant, read, take, greatest, deli...
1720     [story, line, drug, like, book, much, mystery,...
9120     [read, several, angel, book, one, work, didnt,...
360      [possibly, worst, book, ever, read, beginning,...
                               ...                        
1195     [enjoyed, read, think, fan, humorous, must, er...
11877    [pleasantly, surprised, book, enjoyed, m, dubo...
5421     [love, best, friend, since, 15, year, old, 30,...
3855     [fascinating, book, enough, twist, turn, keep,...
4414     [plot, noted, publisher, blurb, publisher, mak...
Name: reviewText, Length: 2400, dtype: object

In [36]:
model=gensim.models.Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [37]:
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)
    
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)
                #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [38]:
from tqdm import tqdm

In [39]:
X_train_vec = np.array([avg_word2vec(sentence) for sentence in X_train])
X_test_vec = np.array([avg_word2vec(sentence) for sentence in X_test])

In [40]:
X_train_vec.shape

(9600, 100)

In [41]:
y_train.shape

(9600,)

In [42]:
X_test_vec

array([[ 0.02879939,  0.2041688 , -0.05169385, ..., -0.45770258,
         0.13981721, -0.11303566],
       [ 0.15614234,  0.25614622,  0.0376687 , ..., -0.63935834,
         0.19019149,  0.04617628],
       [-0.06517132,  0.30648616,  0.2726973 , ..., -0.3707203 ,
         0.03113137, -0.170021  ],
       ...,
       [-0.23885085,  0.37765074, -0.14998022, ..., -0.67034274,
         0.236561  ,  0.05041591],
       [-0.00255704,  0.2637945 ,  0.02968742, ..., -0.58226985,
         0.05562155, -0.01289809],
       [-0.15585999,  0.29037073,  0.13304155, ..., -0.45701012,
         0.13896023, -0.16467835]], dtype=float32)

In [43]:
X_test.shape

(2400,)

In [44]:
y_test.shape

(2400,)

In [45]:
from sklearn.naive_bayes import MultinomialNB

In [46]:
from sklearn.ensemble import RandomForestClassifier

In [47]:
classifier=RandomForestClassifier()

In [48]:
model=classifier.fit(X_train_vec,y_train)

In [49]:
y_pred=model.predict(X_test_vec)

In [50]:
y_pred

array([1, 0, 1, ..., 1, 1, 1])

In [51]:
y_pred.shape,y_test.shape

((2400,), (2400,))

In [52]:
from sklearn.metrics import classification_report

In [53]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.54      0.60       803
           1       0.79      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.75      0.76      0.75      2400



In [54]:
from sklearn.linear_model import LogisticRegression

In [69]:
model2=LogisticRegression(class_weight='balanced').fit(X_train_vec,y_train)

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [70]:
y_pred2=model2.predict(X_test_vec)

In [71]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.54      0.60       803
           1       0.79      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.75      0.76      0.75      2400



In [58]:
df

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1
...,...,...
11995,valentine cupid vampire- jena ian another vamp...,1
11996,read seven book series apocalypticadventure on...,1
11997,book really wasnt cuppa situation man capturin...,1
11998,tried use charge kindle didnt even register ch...,0


In [59]:
X_train.shape

(9600,)

In [61]:
X_train_vec.shape

(9600, 100)

In [62]:
from sklearn.naive_bayes import GaussianNB

In [63]:
nb=GaussianNB()

In [64]:
model3=nb.fit(X_train_vec,y_train)

In [66]:
y_pred3=model.predict(X_test_vec)

In [73]:
print(classification_report(y_test,y_pred3))

              precision    recall  f1-score   support

           0       0.68      0.54      0.60       803
           1       0.79      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.75      0.76      0.75      2400



In [82]:
from sklearn.metrics import confusion_matrix,accuracy_score,roc_auc_score

In [78]:
print("accuarcy score of  logistic regression",accuracy_score(y_pred2,y_test))
print("accuarcy score of  randomforestclassifier",accuracy_score(y_pred,y_test))
print("accuarcy score of  guassian naives bayes",accuracy_score(y_pred3,y_test))

accuarcy score of  logistic regression 0.7525
accuarcy score of  randomforestclassifier 0.76125
accuarcy score of  guassian naives bayes 0.76125


In [79]:
print("logistic regression",confusion_matrix(y_pred2,y_test))
print("randomforestclassifier",confusion_matrix(y_pred,y_test))
print("guassian naives bayes",confusion_matrix(y_pred3,y_test))

logistic regression [[ 610  401]
 [ 193 1196]]
randomforestclassifier [[ 433  203]
 [ 370 1394]]
guassian naives bayes [[ 433  203]
 [ 370 1394]]


In [81]:
print(confusion_matrix(y_pred2,y_test))

[[ 610  401]
 [ 193 1196]]


In [83]:
print(roc_auc_score(y_pred2,y_test))

0.7322070614172825


In [86]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# y_prob = predicted probabilities from your model
# y_true = actual labels

threshold = 0.35  # lower than default 0.5
y_pred_new = (y_pred2 >= threshold).astype(int)

cm_new = confusion_matrix(y_test, y_pred_new)
precision_new = precision_score(y_test, y_pred_new)
recall_new = recall_score(y_test, y_pred_new)
f1_new = f1_score(y_test, y_pred_new)

print("Confusion Matrix:\n", cm_new)
print(f"Precision: {precision_new:.2f}, Recall: {recall_new:.2f}, F1: {f1_new:.2f}")
print(accuracy_score(y_test,y_pred2))

Confusion Matrix:
 [[ 610  193]
 [ 401 1196]]
Precision: 0.86, Recall: 0.75, F1: 0.80
0.7525
